# LeetCode 251 — Flatten 2D Vector bằng Python

Link đề: https://leetcode.com/problems/flatten-2d-vector/ · Độ khó: **Medium** (Premium) · Chủ đề: Design, Iterator, Array, Two Pointers

Skeleton LeetCode:

```python
class Vector2D:
    def __init__(self, vec: List[List[int]]):
        ...

    def next(self) -> int:
        ...

    def hasNext(self) -> bool:
        ...

# Your Vector2D object will be instantiated and called as such:
# obj = Vector2D(vec)
# param_1 = obj.next()
# param_2 = obj.hasNext()
```

Ghi chú: đề bài viết theo hiểu biết về bài (bài Premium), nên đối chiếu lại nếu có tài khoản Premium; ví dụ và constraints lấy từ file đề có sẵn. Đây là **bài thiết kế lớp (design)**, nên ta tự dựng **khung chạy test** mô phỏng đúng định dạng LeetCode (danh sách tên hàm + danh sách đối số), rồi đối chiếu với một mô hình brute force độc lập.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: thiết kế **iterator** duyệt tuần tự mọi phần tử của một **vector 2 chiều** (danh sách các danh sách), với hai thao tác `next()` (trả phần tử kế tiếp và tiến) và `hasNext()` (còn phần tử không).
2. Biết thiết kế một **lớp có trạng thái** trong Python: `__init__`, thuộc tính `self.…`, phương thức, và đọc **định dạng test của bài design** (`["Vector2D","next",...]`).
3. Đi từ cách **làm phẳng ngay trong `__init__`** → **hai con trỏ `(i, j)` + hàm `forward` bỏ qua hàng rỗng** (lười, `O(1)` bộ nhớ phụ) → **iterator lồng nhau (outer/inner)** như yêu cầu follow-up → **`itertools.chain` + nhìn trước (lookahead)**.
4. Hiểu **vì sao `hasNext` phải bỏ qua các hàng rỗng**, **không được làm mất phần tử** (không tiêu thụ), và khái niệm **chi phí khấu hao (amortized)**.
5. Tránh các lỗi kinh điển: `hasNext` chỉ kiểm tra hàng hiện tại, `next` quên bỏ qua hàng rỗng, `hasNext` làm **tiến** con trỏ, sửa/đụng vào dữ liệu đầu vào (`pop(0)`), và lỗi chỉ số ở hàng rỗng cuối.

## 2. Đề bài & ví dụ

Thiết kế một **iterator** để **làm phẳng** một vector 2 chiều. Nó phải hỗ trợ hai thao tác `next` và `hasNext`.

Cài đặt lớp `Vector2D`:

- `Vector2D(int[][] vec)` khởi tạo đối tượng với vector 2 chiều `vec`.
- `next()` trả về phần tử kế tiếp của vector 2 chiều và **tiến con trỏ một bước**. Có thể giả định mọi lời gọi `next` đều hợp lệ.
- `hasNext()` trả `true` nếu **còn** phần tử, ngược lại `false`.

**Ví dụ 1**

```
Input
["Vector2D", "next", "next", "next", "hasNext", "hasNext", "next", "hasNext"]
[[[[1, 2], [3], [4]]], [], [], [], [], [], [], []]
Output
[null, 1, 2, 3, true, true, 4, false]
```

Giải thích:

```
Vector2D vector2D = new Vector2D([[1, 2], [3], [4]]);
vector2D.next();    // trả 1
vector2D.next();    // trả 2
vector2D.next();    // trả 3
vector2D.hasNext(); // trả true
vector2D.hasNext(); // trả true
vector2D.next();    // trả 4
vector2D.hasNext(); // trả false
```

**Ràng buộc:** `0 <= vec.length <= 200`; `0 <= vec[i].length <= 500`; `-500 <= vec[i][j] <= 500`; tối đa `10^5` lời gọi `next` và `hasNext`.

**Follow up:** thử cài đặt chỉ bằng **iterator** (C++/Java). Trong Python ta tương đương dùng `iter()`/`next()` hoặc `itertools`.

### Khung chạy test kiểu LeetCode

Với bài design, LeetCode đưa hai danh sách: `ops` (tên thao tác) và `args` (đối số). Ta viết hàm `run(cls, ops, args)`: phần tử đầu là tên lớp (tạo đối tượng), các phần tử sau gọi phương thức tương ứng; kết quả `None` cho hàm tạo (hiển thị `null`).

In [1]:
from typing import List
from itertools import chain
import random
import time

examples_ops = ["Vector2D", "next", "next", "next", "hasNext", "hasNext", "next", "hasNext"]
examples_args = [[[[1, 2], [3], [4]]], [], [], [], [], [], [], []]
examples_expected = [None, 1, 2, 3, True, True, 4, False]


def run(cls, ops, args):
    """Chạy một chuỗi thao tác kiểu LeetCode. ops[0] là tên lớp (hàm tạo), các ops sau là tên phương thức."""
    obj = None
    out = []
    for op, arg in zip(ops, args):
        if obj is None:
            obj = cls(*arg)
            out.append(None)
        else:
            out.append(getattr(obj, op)(*arg))
    return out


print("ops     :", examples_ops)
print("args    :", examples_args)
print("kỳ vọng :", examples_expected)

ops     : ['Vector2D', 'next', 'next', 'next', 'hasNext', 'hasNext', 'next', 'hasNext']
args    : [[[[1, 2], [3], [4]]], [], [], [], [], [], [], []]
kỳ vọng : [None, 1, 2, 3, True, True, 4, False]


## 3. Kiến thức nền

### 3.1. Lớp (class), `self` và thuộc tính

Một **lớp** gom dữ liệu (thuộc tính) và hành vi (phương thức). `__init__(self, ...)` chạy khi tạo đối tượng; `self.x` là thuộc tính riêng của từng đối tượng, **giữ trạng thái giữa các lần gọi**. Với iterator, trạng thái là "đang đứng ở đâu".
#### Code demo

In [2]:
class Counter:
    def __init__(self, start):
        self.value = start               # trạng thái riêng của đối tượng

    def next(self):
        self.value += 1                  # thay đổi trạng thái
        return self.value


c = Counter(10)
print(c.next(), c.next(), c.value)       # 11 12 12
d = Counter(0)
print(d.next(), c.value)                 # 1 12: hai đối tượng độc lập

11 12 12
1 12


### 3.2. Vector 2 chiều và chỉ số `(i, j)`

`vec[i]` là hàng `i`, `vec[i][j]` là phần tử thứ `j` của hàng `i`. Hàng có thể **rỗng** (`[]`), và `vec` cũng có thể rỗng. Một **con trỏ** `(i, j)` chỉ vào phần tử kế tiếp cần trả; nếu `j` đã chạm cuối hàng `i` thì phải **sang hàng tiếp theo** (có thể phải sang liền nhiều hàng nếu có hàng rỗng).
#### Code demo

In [3]:
vec = [[], [1, 2], [], [3], []]
for i, row in enumerate(vec):
    print(f"hàng {i}: {row} (độ dài {len(row)})")
print(vec[1][0], vec[3][0])      # 1 3

hàng 0: [] (độ dài 0)
hàng 1: [1, 2] (độ dài 2)
hàng 2: [] (độ dài 0)
hàng 3: [3] (độ dài 1)
hàng 4: [] (độ dài 0)
1 3


### 3.3. Iterator trong Python: `iter()` và `next(it, default)`

`iter(list)` tạo một **iterator**; `next(it)` trả phần tử kế tiếp và tiến; hết thì ném `StopIteration`. Dạng `next(it, mặc_định)` trả `mặc_định` thay vì ném lỗi. Iterator **chỉ đi tiến**, nên để biết "còn phần tử không" ta phải **nhìn trước (peek)**: đọc sẵn phần tử tiếp theo vào một biến đệm.
#### Code demo

In [4]:
it = iter([1, 2])
print(next(it), next(it))                 # 1 2
print(next(it, "hết"))                    # hết (không ném lỗi)
END = object()                            # "lính canh": giá trị đặc biệt không thể nhầm với dữ liệu
print(next(iter([]), END) is END)         # True

1 2
hết
True


### 3.4. `itertools.chain.from_iterable` — nối nhiều danh sách

`chain.from_iterable([[1, 2], [3], [], [4]])` là một iterator duyệt `1, 2, 3, 4` — **tự bỏ qua hàng rỗng** và **không tạo bản sao** toàn bộ dữ liệu. Ta có thể dùng nó làm "nguồn" rồi chỉ cần thêm một biến nhìn trước để có `hasNext`.
#### Code demo

In [5]:
print(list(chain.from_iterable([[1, 2], [3], [], [4]])))     # [1, 2, 3, 4]
print(list(chain.from_iterable([[], []])))                    # []

[1, 2, 3, 4]
[]


### 3.5. Chi phí khấu hao (amortized)

Một thao tác có thể **lúc nhanh lúc chậm** nhưng **trung bình** mỗi lời gọi tốn `O(1)`. Ví dụ `hasNext` có thể phải bỏ qua nhiều hàng rỗng một lần, nhưng **mỗi hàng chỉ bị bỏ qua đúng một lần** trong suốt vòng đời iterator; tổng công việc bỏ qua `<= số hàng`, chia đều cho các lời gọi.
#### Code demo

In [6]:
rows = [[] for _ in range(5)] + [[1]]
skipped = 0
i = 0
while i < len(rows) and not rows[i]:
    skipped += 1
    i += 1
print("lần đầu bỏ qua", skipped, "hàng rỗng; sau đó không phải bỏ qua nữa")

lần đầu bỏ qua 5 hàng rỗng; sau đó không phải bỏ qua nữa


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Với `vec = [[1, 2], [3], [4]]`, hình dung một ngón tay chỉ vào một ô:

- ngón tay ở `(0,0)` → `next()` trả `1`, tiến sang `(0,1)`;
- `next()` trả `2`, tiến sang `(0,2)` — nhưng hàng `0` chỉ có `2` phần tử nên **thực chất** ngón tay phải nhảy sang `(1,0)`;
- `next()` trả `3`, tiến, rồi nhảy `(2,0)`; `next()` trả `4`, sau đó hết.

Với `vec = [[], [1], [], []]`: ngón tay bắt đầu `(0,0)`, nhưng hàng `0` rỗng nên phải **nhảy** sang hàng `1`: `hasNext()` ngay từ đầu là `True`, `next()` trả `1`, sau đó mọi hàng còn lại rỗng nên `hasNext()` là `False`.

**Cách nghĩ then chốt:** "**đưa ngón tay tới một ô có thật**" (bỏ qua mọi hàng đã hết) trước mỗi lần trả lời; việc này nên làm trong **cả `hasNext` và `next`**, nhưng **không được làm mất phần tử**.

## 5. Cách 1 — Brute Force / baseline: làm phẳng ngay trong hàm khởi tạo

**Ý tưởng.** Ở `__init__`, duyệt mọi hàng và gom mọi phần tử vào một danh sách `self.flat`; giữ con trỏ `self.pos = 0`. `next()` trả `flat[pos]` rồi `pos += 1`; `hasNext()` kiểm tra `pos < len(flat)`.
#### Code demo

In [7]:
class Vector2DFlatten:
    def __init__(self, vec: List[List[int]]):
        self.flat = [x for row in vec for x in row]     # làm phẳng một lần
        self.pos = 0

    def next(self) -> int:
        value = self.flat[self.pos]
        self.pos += 1
        return value

    def hasNext(self) -> bool:
        return self.pos < len(self.flat)


print(run(Vector2DFlatten, examples_ops, examples_args), "(kỳ vọng", examples_expected, ")")

[None, 1, 2, 3, True, True, 4, False] (kỳ vọng [None, 1, 2, 3, True, True, 4, False] )


**Độ phức tạp (Cách 1).** Gọi `N` là tổng số phần tử, `R` số hàng. Khởi tạo **`O(N + R)`** thời gian và **`O(N)`** bộ nhớ phụ (một bản sao toàn bộ); `next` và `hasNext` đều **`O(1)`**.

**Nút thắt.** Ta copy hết dữ liệu dù người dùng có thể chỉ gọi `next` vài lần (hoặc dữ liệu rất lớn/đang thay đổi). Phiên bản **lười (lazy)** chỉ đi qua dữ liệu khi cần và **không copy**.

## 6. Ý tưởng thuật toán / bước đột phá

**Làm lười bằng hai con trỏ.** Giữ `(i, j)` = vị trí phần tử kế tiếp. Viết một hàm phụ `forward()`: **trong khi** `i` còn hợp lệ và `j` đã hết hàng `i` (`j >= len(vec[i])`), tăng `i` và đặt `j = 0`. Sau `forward()`, hoặc `i == len(vec)` (**hết**) hoặc `vec[i][j]` là phần tử thật.

- `hasNext()`: gọi `forward()` rồi trả `i < len(vec)`.
- `next()`: gọi `forward()`, đọc `vec[i][j]`, `j += 1`.

`forward()` chỉ **di chuyển ngón tay tới ô thật tiếp theo** chứ không tiêu thụ giá trị, nên `hasNext` gọi nhiều lần không làm mất gì; mỗi hàng bị bỏ qua tối đa một lần ⇒ khấu hao `O(1)`.

## 7. Các cách tiếp theo

### Cách 2 — Hai con trỏ `(i, j)` với `forward()` (cách nộp chuẩn)

**Giải thích.** Trạng thái: `self.vec`, `self.i`, `self.j`. Không copy dữ liệu.
#### Code demo

In [8]:
class Vector2DPointers:
    def __init__(self, vec: List[List[int]]):
        self.vec = vec
        self.i = 0                         # chỉ số hàng
        self.j = 0                         # chỉ số cột trong hàng

    def _forward(self):
        # bỏ qua mọi hàng đã hết (kể cả hàng rỗng) cho tới khi gặp ô thật hoặc hết dữ liệu
        while self.i < len(self.vec) and self.j >= len(self.vec[self.i]):
            self.i += 1
            self.j = 0

    def next(self) -> int:
        self._forward()
        value = self.vec[self.i][self.j]
        self.j += 1
        return value

    def hasNext(self) -> bool:
        self._forward()
        return self.i < len(self.vec)


print(run(Vector2DPointers, examples_ops, examples_args), "(kỳ vọng", examples_expected, ")")

[None, 1, 2, 3, True, True, 4, False] (kỳ vọng [None, 1, 2, 3, True, True, 4, False] )


In [9]:
# Code nộp LeetCode cho Cách 2
class Vector2D:
    def __init__(self, vec: List[List[int]]):
        self.vec = vec
        self.i = 0
        self.j = 0

    def next(self) -> int:
        self.forward()
        ans = self.vec[self.i][self.j]
        self.j += 1
        return ans

    def hasNext(self) -> bool:
        self.forward()
        return self.i < len(self.vec)

    def forward(self):
        while self.i < len(self.vec) and self.j >= len(self.vec[self.i]):
            self.i += 1
            self.j = 0


print(run(Vector2D, examples_ops, examples_args))

[None, 1, 2, 3, True, True, 4, False]


**Độ phức tạp Cách 2:** `__init__` **`O(1)`**; `next`/`hasNext` **`O(1)` khấu hao** (tổng bỏ qua hàng `<= R`); **Space `O(1)`** bộ nhớ phụ (chỉ vài biến, không copy dữ liệu).

### Cách 3 — Hai iterator lồng nhau (outer / inner) với biến nhìn trước

**Giải thích.** Đúng tinh thần follow-up: `outer = iter(vec)` đi qua các hàng, `inner = iter(hàng hiện tại)` đi qua phần tử. Vì iterator không cho biết còn phần tử không, ta giữ **một ô đệm `peeked`**: `_fill()` đảm bảo `peeked` chứa phần tử kế tiếp (nếu còn), bằng cách lấy từ `inner`, hết thì mở hàng mới từ `outer`.
#### Code demo

In [10]:
_END = object()                                   # lính canh (sentinel)


class Vector2DIterators:
    def __init__(self, vec: List[List[int]]):
        self.outer = iter(vec)
        self.inner = iter(())                     # iterator rỗng ban đầu
        self.peeked = _END

    def _fill(self):
        while self.peeked is _END:
            value = next(self.inner, _END)
            if value is not _END:
                self.peeked = value
                return
            row = next(self.outer, _END)
            if row is _END:
                return                            # hết hẳn dữ liệu
            self.inner = iter(row)

    def next(self) -> int:
        self._fill()
        value = self.peeked
        self.peeked = _END                        # tiêu thụ phần tử đã đọc trước
        return value

    def hasNext(self) -> bool:
        self._fill()
        return self.peeked is not _END


print(run(Vector2DIterators, examples_ops, examples_args), "(kỳ vọng", examples_expected, ")")

[None, 1, 2, 3, True, True, 4, False] (kỳ vọng [None, 1, 2, 3, True, True, 4, False] )


**Độ phức tạp Cách 3:** `next`/`hasNext` **`O(1)` khấu hao**, **Space `O(1)`** (hai iterator và một ô đệm).

### Cách 4 — `itertools.chain` + nhìn trước

**Giải thích.** `chain.from_iterable(vec)` đã nối mọi hàng lại thành **một** iterator (tự bỏ hàng rỗng). Ta chỉ cần ô đệm `peeked` để trả lời `hasNext`.
#### Code demo

In [11]:
class Vector2DChain:
    def __init__(self, vec: List[List[int]]):
        self.it = chain.from_iterable(vec)
        self.peeked = _END

    def _fill(self):
        if self.peeked is _END:
            self.peeked = next(self.it, _END)

    def next(self) -> int:
        self._fill()
        value = self.peeked
        self.peeked = _END
        return value

    def hasNext(self) -> bool:
        self._fill()
        return self.peeked is not _END


print(run(Vector2DChain, examples_ops, examples_args), "(kỳ vọng", examples_expected, ")")

[None, 1, 2, 3, True, True, 4, False] (kỳ vọng [None, 1, 2, 3, True, True, 4, False] )


**Độ phức tạp Cách 4:** `next`/`hasNext` **`O(1)` khấu hao**, **Space `O(1)`**.

> Cách 2 là bản nộp chuẩn, dễ kiểm soát. Cách 3 và 4 minh họa tư duy "iterator + lookahead".

## 8. TRACE từng bước

Ta in trạng thái `(i, j)` trước/sau từng lời gọi của Cách 2 với hai dữ liệu: ví dụ của đề và một vector có nhiều hàng rỗng.
#### Code demo

In [12]:
def trace_pointers(vec, ops):
    print("vec =", vec)
    obj = Vector2DPointers(vec)
    for op in ops:
        before = (obj.i, obj.j)
        result = getattr(obj, op)()
        after = (obj.i, obj.j)
        print(f"{op:>8}() -> {str(result):>5}   (i, j): {before} -> {after}")


trace_pointers([[1, 2], [3], [4]], ["next", "next", "next", "hasNext", "hasNext", "next", "hasNext"])
print()
trace_pointers([[], [1], [], [], [2, 3], []], ["hasNext", "hasNext", "next", "hasNext", "next", "next", "hasNext"])

vec = [[1, 2], [3], [4]]
    next() ->     1   (i, j): (0, 0) -> (0, 1)
    next() ->     2   (i, j): (0, 1) -> (0, 2)
    next() ->     3   (i, j): (0, 2) -> (1, 1)
 hasNext() ->  True   (i, j): (1, 1) -> (2, 0)
 hasNext() ->  True   (i, j): (2, 0) -> (2, 0)
    next() ->     4   (i, j): (2, 0) -> (2, 1)
 hasNext() -> False   (i, j): (2, 1) -> (3, 0)

vec = [[], [1], [], [], [2, 3], []]
 hasNext() ->  True   (i, j): (0, 0) -> (1, 0)
 hasNext() ->  True   (i, j): (1, 0) -> (1, 0)
    next() ->     1   (i, j): (1, 0) -> (1, 1)
 hasNext() ->  True   (i, j): (1, 1) -> (4, 0)
    next() ->     2   (i, j): (4, 0) -> (4, 1)
    next() ->     3   (i, j): (4, 1) -> (4, 2)
 hasNext() -> False   (i, j): (4, 2) -> (6, 0)


Bảng tóm tắt (`vec = [[1, 2], [3], [4]]`):

| lời gọi | `(i, j)` trước | `forward` làm gì | trả về | `(i, j)` sau |
|---|---|---|---|---|
| `next` | `(0,0)` | không | `1` | `(0,1)` |
| `next` | `(0,1)` | không | `2` | `(0,2)` |
| `next` | `(0,2)` | `j=2 >= len=2`: sang `(1,0)` | `3` | `(1,1)` |
| `hasNext` | `(1,1)` | `j=1 >= len=1`: sang `(2,0)` | `True` | `(2,0)` |
| `hasNext` | `(2,0)` | không | `True` | `(2,0)` (không đổi) |
| `next` | `(2,0)` | không | `4` | `(2,1)` |
| `hasNext` | `(2,1)` | sang `(3,0)` | `False` | `(3,0)` |

Chú ý hai lời gọi `hasNext` liên tiếp **không làm mất phần tử**: `forward` chỉ đưa con trỏ tới ô thật, không đọc/tiêu thụ.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 2 (hai con trỏ). Đúng tên lớp `Vector2D`, hàm tạo `__init__(self, vec)`, hai phương thức `next()` và `hasNext()`.

In [13]:
class Vector2D:
    def __init__(self, vec: List[List[int]]):
        self.vec = vec
        self.i = 0                       # hàng hiện tại
        self.j = 0                       # cột hiện tại

    def next(self) -> int:
        self.forward()                   # bảo đảm (i, j) trỏ tới một ô thật (đề đảm bảo next hợp lệ)
        ans = self.vec[self.i][self.j]
        self.j += 1
        return ans

    def hasNext(self) -> bool:
        self.forward()                   # bỏ qua các hàng đã hết / hàng rỗng
        return self.i < len(self.vec)

    def forward(self):
        while self.i < len(self.vec) and self.j >= len(self.vec[self.i]):
            self.i += 1
            self.j = 0


assert run(Vector2D, examples_ops, examples_args) == examples_expected
print("OK: ví dụ của đề")

OK: ví dụ của đề


## 10. Vì sao thuật toán đúng

- **Invariant sau `forward()`:** hoặc `i == len(vec)` (đã hết), hoặc `j < len(vec[i])` (con trỏ chỉ vào một phần tử thật chưa trả). Vòng `while` chỉ dừng khi một trong hai điều kiện đúng.
- **`hasNext` đúng:** sau `forward()`, còn phần tử ⟺ `i < len(vec)`; và `forward` **không đọc, không làm tăng `j` trên ô thật**, nên gọi lặp lại không đổi gì (idempotent).
- **`next` đúng:** sau `forward()` (và với giả định `next` hợp lệ) `vec[i][j]` là phần tử kế tiếp theo thứ tự hàng rồi cột; tăng `j` để lần sau chuyển tiếp.
- **Thứ tự duyệt:** hàng theo `i` tăng, trong hàng theo `j` tăng — đúng thứ tự "làm phẳng" của đề; hàng rỗng không đóng góp phần tử.
- **Khấu hao:** `i` chỉ tăng (không bao giờ giảm) và mỗi lần tăng tương ứng một hàng bị bỏ qua, nên tổng số lần lặp của `while` qua mọi lời gọi `<= R + (số lần gọi)`.
- **Cách 3, 4** giữ cùng ý nhưng dùng iterator: ô đệm `peeked` đóng vai "ô thật tiếp theo", `_fill` giống `forward`.

## 11. Phân tích Time / Space Complexity

Gọi `N` = tổng số phần tử, `R` = số hàng, `C` = số lời gọi.

| Cách | `__init__` | `next` / `hasNext` | Space phụ |
|---|---|---|---|
| 1. Làm phẳng sẵn | `O(N + R)` | `O(1)` | `O(N)` |
| 2. Hai con trỏ | `O(1)` | `O(1)` khấu hao | `O(1)` |
| 3. Hai iterator | `O(1)` | `O(1)` khấu hao | `O(1)` |
| 4. `chain` + lookahead | `O(1)` | `O(1)` khấu hao | `O(1)` |

Tổng công việc cho `C` lời gọi của Cách 2–4 là `O(C + R)`; với `R <= 200`, `C <= 10^5` rất nhẹ. Worst case một lời gọi đơn lẻ `hasNext` có thể tốn `O(R)` (khi toàn hàng rỗng liền nhau) nhưng không lặp lại ở các lời gọi sau.

## 12. Bộ test

`ModelVector2D` là **mô hình brute force độc lập**: tính sẵn danh sách `flat = [mọi phần tử theo thứ tự]` bằng hai vòng `for`, giữ con trỏ `pos`. Với chuỗi thao tác ngẫu nhiên hợp lệ, ta so sánh **đầu ra từng bước** của mọi cách với mô hình.
#### Code demo

In [14]:
def model_run(vec, ops):
    """Kết quả kỳ vọng cho dãy thao tác ops (list tên 'next'/'hasNext'), tính trực tiếp trên danh sách phẳng."""
    flat = []
    for row in vec:
        for value in row:
            flat.append(value)
    pos = 0
    out = [None]
    for op in ops:
        if op == "hasNext":
            out.append(pos < len(flat))
        else:
            out.append(flat[pos])
            pos += 1
    return out


def random_ops(vec, rng, max_extra=3):
    """Dãy thao tác hợp lệ: chỉ gọi next khi còn phần tử (theo mô hình); xen kẽ hasNext ngẫu nhiên."""
    total = sum(len(row) for row in vec)
    taken = 0
    ops = []
    while taken < total or rng.random() < 0.5:
        if rng.random() < 0.4 or taken == total:
            ops.append("hasNext")
        else:
            ops.append("next")
            taken += 1
        if taken == total and len(ops) > 3 * total + max_extra:
            break
        if taken == total and rng.random() < 0.3:
            break
    return ops


impls = {
    "flatten": Vector2DFlatten,
    "pointers": Vector2DPointers,
    "iterators": Vector2DIterators,
    "chain": Vector2DChain,
    "submit": Vector2D,
}

for key, cls in impls.items():
    assert run(cls, examples_ops, examples_args) == examples_expected, key
assert model_run([[1, 2], [3], [4]], examples_ops[1:]) == examples_expected
print("PASS ví dụ của đề (mọi cách và mô hình)")

PASS ví dụ của đề (mọi cách và mô hình)


### 12.1. Các trường hợp biên
#### Code demo

In [15]:
edge_cases = [
    ([], ["hasNext"], "vector rỗng: hasNext = False"),
    ([[]], ["hasNext"], "một hàng rỗng"),
    ([[], [], []], ["hasNext", "hasNext"], "toàn hàng rỗng"),
    ([[1]], ["hasNext", "next", "hasNext"], "một phần tử"),
    ([[], [1], []], ["hasNext", "next", "hasNext"], "phần tử nằm giữa hàng rỗng"),
    ([[1, 2, 3]], ["next", "next", "next", "hasNext"], "một hàng"),
    ([[1], [2], [3]], ["next", "hasNext", "next", "hasNext", "next", "hasNext"], "mỗi hàng một phần tử"),
    ([[], [], [5], [], [], [6, 7], []], ["hasNext", "next", "hasNext", "next", "next", "hasNext"], "nhiều hàng rỗng xen kẽ"),
    ([[-500, 500]], ["next", "next", "hasNext"], "giá trị biên của đề"),
    ([[1, 2], [3]], ["hasNext", "hasNext", "hasNext", "next"], "hasNext gọi nhiều lần không làm mất phần tử"),
]
for vec, ops, label in edge_cases:
    expected = model_run(vec, ops)
    args = [[vec]] + [[] for _ in ops]
    for key, cls in impls.items():
        got = run(cls, ["Vector2D"] + ops, args)
        assert got == expected, (key, label, got, expected)
    print("PASS", label, "->", expected)

PASS vector rỗng: hasNext = False -> [None, False]
PASS một hàng rỗng -> [None, False]
PASS toàn hàng rỗng -> [None, False, False]
PASS một phần tử -> [None, True, 1, False]
PASS phần tử nằm giữa hàng rỗng -> [None, True, 1, False]
PASS một hàng -> [None, 1, 2, 3, False]
PASS mỗi hàng một phần tử -> [None, 1, True, 2, True, 3, False]
PASS nhiều hàng rỗng xen kẽ -> [None, True, 5, True, 6, 7, False]
PASS giá trị biên của đề -> [None, -500, 500, False]
PASS hasNext gọi nhiều lần không làm mất phần tử -> [None, True, True, True, 1]


### 12.2. Tính chất và phản ví dụ

- **`hasNext` không có tác dụng phụ:** chèn bao nhiêu `hasNext` vào chuỗi cũng không đổi dãy giá trị của các `next`.
- Tổng số `next` thành công = tổng số phần tử; sau đó `hasNext` luôn `False`.
- **Phản ví dụ** cho bản `hasNext` chỉ kiểm tra hàng hiện tại: với `[[], [1]]` trả `False` ngay từ đầu (sai).
#### Code demo

In [16]:
def sai_hasnext_hang_hien_tai(vec):
    class Wrong:
        def __init__(self, vec):
            self.vec, self.i, self.j = vec, 0, 0

        def hasNext(self):
            return self.i < len(self.vec) and self.j < len(self.vec[self.i])      # SAI: không nhảy qua hàng rỗng

    return Wrong(vec).hasNext()


print("[[], [1]] đúng:", run(Vector2D, ["Vector2D", "hasNext"], [[[[], [1]]], []])[1], "| bản sai:", sai_hasnext_hang_hien_tai([[], [1]]))
assert sai_hasnext_hang_hien_tai([[], [1]]) is False

rng = random.Random(251)
for _ in range(300):
    vec = [[rng.randint(-5, 5) for _ in range(rng.randint(0, 4))] for _ in range(rng.randint(0, 6))]
    ops_plain = [op for op in random_ops(vec, rng) if op == "next"]          # chỉ các next, theo thứ tự
    total = sum(len(r) for r in vec)
    ops_with_has = []
    for op in ops_plain:
        ops_with_has += ["hasNext"] * rng.randint(0, 3) + [op]
    values_plain = [v for v in model_run(vec, ops_plain)[1:]]
    values_has = [v for op, v in zip(ops_with_has, model_run(vec, ops_with_has)[1:]) if op == "next"]
    assert values_plain == values_has
    got = run(Vector2D, ["Vector2D"] + ops_with_has + ["hasNext"], [[vec]] + [[] for _ in ops_with_has] + [[]])
    assert got[-1] is (len(ops_plain) < total)
print("PASS tính chất: hasNext không có tác dụng phụ (300 vector ngẫu nhiên)")

[[], [1]] đúng: True | bản sai: False
PASS tính chất: hasNext không có tác dụng phụ (300 vector ngẫu nhiên)


### 12.3. Đối chiếu ngẫu nhiên (có seed) và dữ liệu lớn
#### Code demo

In [17]:
rng = random.Random(4251)
for trial in range(1500):
    vec = [[rng.randint(-500, 500) for _ in range(rng.randint(0, 5))] for _ in range(rng.randint(0, 8))]
    ops = random_ops(vec, rng)
    expected = model_run(vec, ops)
    args = [[[list(r) for r in vec]]] + [[] for _ in ops]
    for key, cls in impls.items():
        got = run(cls, ["Vector2D"] + ops, args)
        assert got == expected, (key, vec, ops, got, expected)
print("PASS 1500 vector ngẫu nhiên: mọi cách khớp mô hình theo từng bước")

# Dữ liệu cỡ đề: 200 hàng x tối đa 500, khoảng 10^5 lời gọi
rng = random.Random(100_000)
vec = [[rng.randint(-500, 500) for _ in range(rng.choice([0, 0, 500, 100]))] for _ in range(200)]
total = sum(len(r) for r in vec)
ops, taken = [], 0
while len(ops) < 100_000 and taken < total:
    if rng.random() < 0.5:
        ops.append("hasNext")
    else:
        ops.append("next")
        taken += 1
expected = model_run(vec, ops)
for key, cls in impls.items():
    t0 = time.perf_counter()
    got = run(cls, ["Vector2D"] + ops, [[vec]] + [[] for _ in ops])
    dt = (time.perf_counter() - t0) * 1000
    assert got == expected, key
    print(f"{key:<10} {len(ops)} lời gọi, {total} phần tử: {dt:7.1f} ms")
print("PASS dữ liệu cỡ đề")

PASS 1500 vector ngẫu nhiên: mọi cách khớp mô hình theo từng bước
flatten    60842 lời gọi, 30400 phần tử:    15.5 ms
pointers   60842 lời gọi, 30400 phần tử:    17.7 ms


iterators  60842 lời gọi, 30400 phần tử:    34.3 ms


chain      60842 lời gọi, 30400 phần tử:    13.6 ms


submit     60842 lời gọi, 30400 phần tử:    16.0 ms
PASS dữ liệu cỡ đề


## 13. Các lỗi người mới hay gặp

### Lỗi 1: `hasNext` chỉ kiểm tra hàng hiện tại (không bỏ qua hàng rỗng)
Xem `sai_hasnext_hang_hien_tai` ở phần test: với `[[], [1]]` trả `False` dù còn phần tử `1`. **Code đúng:** `hasNext` phải gọi `forward()` (nhảy qua hàng rỗng) rồi mới kiểm tra.

### Lỗi 2: `next` quên nhảy qua hàng rỗng / hàng đã hết
#### Code sai

In [18]:
class SaiNextKhongForward:
    def __init__(self, vec):
        self.vec, self.i, self.j = vec, 0, 0

    def next(self):
        value = self.vec[self.i][self.j]              # SAI: không forward, hàng i có thể rỗng hoặc đã hết
        self.j += 1
        return value

    def hasNext(self):
        while self.i < len(self.vec) and self.j >= len(self.vec[self.i]):
            self.i += 1
            self.j = 0
        return self.i < len(self.vec)


for vec, ops in [([[1, 2], [3]], ["next", "next", "next"]), ([[], [1]], ["next"])]:
    try:
        w = SaiNextKhongForward(vec)
        print(vec, [w.next() for _ in ops], "| đúng:", model_run(vec, ops)[1:])
    except IndexError as e:
        print(vec, "-> IndexError:", e, "| đúng:", model_run(vec, ops)[1:])

[[1, 2], [3]] -> IndexError: list index out of range | đúng: [1, 2, 3]
[[], [1]] -> IndexError: list index out of range | đúng: [1]


**Vì sao sai:** chỉ gọi `hasNext` trước `next` mới nhảy hàng; nhưng đề **không bắt buộc** gọi `hasNext` trước mỗi `next` (ví dụ của đề gọi `next` liên tiếp). **Code đúng:** `next` cũng phải gọi `forward()`.

### Lỗi 3: `hasNext` làm mất phần tử (tiêu thụ)
#### Code sai

In [19]:
class SaiHasNextTieuThu:
    def __init__(self, vec):
        self.it = chain.from_iterable(vec)

    def hasNext(self):
        try:
            next(self.it)                              # SAI: đọc và vứt luôn phần tử
            return True
        except StopIteration:
            return False

    def next(self):
        return next(self.it)


w = SaiHasNextTieuThu([[1, 2], [3]])
print("hasNext:", w.hasNext(), "-> next trả:", w.next(), "(đúng phải là 1)")

hasNext: True -> next trả: 2 (đúng phải là 1)


**Vì sao sai:** iterator chỉ tiến; đọc để kiểm tra rồi không cất lại thì mất phần tử. **Code đúng:** lưu phần tử đọc trước vào ô đệm `peeked` (Cách 3, 4) hoặc dùng con trỏ chỉ số (Cách 2).

### Lỗi 4: Sửa dữ liệu đầu vào (`pop(0)`, `del`)
#### Code sai

In [20]:
class SaiPop:
    def __init__(self, vec):
        self.vec = vec

    def hasNext(self):
        while self.vec and not self.vec[0]:
            self.vec.pop(0)                           # SAI: sửa dữ liệu của người gọi, và pop(0) tốn O(R)
        return bool(self.vec)

    def next(self):
        self.hasNext()
        return self.vec[0].pop(0)                     # SAI: pop(0) tốn O(độ dài hàng)


original = [[1, 2], [], [3]]
w = SaiPop(original)
print([w.next(), w.next(), w.next()], "| dữ liệu gốc sau khi dùng:", original)

[1, 2, 3] | dữ liệu gốc sau khi dùng: [[]]


**Vì sao sai:** phá hủy đầu vào (người gọi mất dữ liệu) và `pop(0)` chậm `O(n)` mỗi lần. **Code đúng:** chỉ **đọc** dữ liệu và dịch **chỉ số**/iterator.

### Lỗi 5: Hàng rỗng ở cuối gây `IndexError`
#### Code sai

In [21]:
class SaiHangRongCuoi:
    def __init__(self, vec):
        self.vec, self.i, self.j = vec, 0, 0

    def hasNext(self):
        if self.j >= len(self.vec[self.i]):            # SAI: nhảy tối đa một hàng, và không kiểm tra i hợp lệ
            self.i += 1
            self.j = 0
        return self.i < len(self.vec)

    def next(self):
        self.hasNext()
        value = self.vec[self.i][self.j]
        self.j += 1
        return value


for vec in ([[1], [], []], [[], [], [1]]):
    try:
        w = SaiHangRongCuoi(vec)
        print(vec, "hasNext lần 1:", w.hasNext(), "next:", w.next(), "hasNext lần 2:", w.hasNext())
    except IndexError as e:
        print(vec, "-> IndexError:", e)

[[1], [], []] hasNext lần 1: True next: 1 hasNext lần 2: True
[[], [], [1]] hasNext lần 1: True next: 1 hasNext lần 2: False


**Vì sao sai:** dùng `if` (nhảy tối đa **một** hàng) thay vì `while`; và `self.vec[self.i]` truy cập khi `i` có thể bằng `len(vec)`. **Code đúng:** `while self.i < len(self.vec) and self.j >= len(self.vec[self.i])` — điều kiện biên đặt **trước** khi đọc.

## 14. Mẹo làm nhanh & nhận dạng pattern

- Thấy "**iterator / flatten / next & hasNext**" → trạng thái là **con trỏ**; làm một hàm "**tiến tới ô thật tiếp theo**" rồi gọi ở cả `next` và `hasNext`.
- **`hasNext` phải idempotent** (gọi nhiều lần giống một lần) và **không tiêu thụ**: đưa con trỏ tới chỗ cần thiết nhưng không đọc/bỏ phần tử.
- Dùng **`while`** (không phải `if`) để bỏ qua **nhiều** hàng rỗng liền nhau; đặt điều kiện biên **trước** khi truy cập `vec[i]`.
- Với **iterator thuần** (không có chỉ số): dùng **ô đệm nhìn trước** (`peeked`) + **sentinel** (`object()`), như Java `PeekingIterator`.
- Khi dữ liệu lớn hoặc đổi trong lúc duyệt: **lười** (lazy) tốt hơn làm phẳng sẵn (tốn bộ nhớ, không phản ánh thay đổi).
- Test bài design: tự viết `run(cls, ops, args)` và **mô hình brute force** để so từng bước với chuỗi thao tác ngẫu nhiên.

## 15. Biến thể của bài

1. **Flatten Nested List Iterator** (LeetCode 341): lồng nhiều tầng bất kỳ — dùng stack/đệ quy lười.
2. **Peeking Iterator** (284): thêm `peek()` trên một iterator — chính là ô đệm của Cách 3/4.
3. **Zigzag Iterator** (281): luân phiên lấy từ hai (hoặc `k`) danh sách.
4. **Vector2D có `remove()`** (Java-style `Iterator.remove`): xóa phần tử vừa trả.
5. **Duyệt ngược** hoặc **theo cột**: đổi quy tắc duyệt.
#### Code demo

In [22]:
class ZigzagIterator:
    """Biến thể (LeetCode 281): luân phiên lấy từ nhiều danh sách bằng hàng đợi các iterator."""
    def __init__(self, *lists):
        from collections import deque
        self.queue = deque(iter(lst) for lst in lists if lst)
        self.peeked = _END

    def _fill(self):
        while self.peeked is _END and self.queue:
            it = self.queue.popleft()
            value = next(it, _END)
            if value is not _END:
                self.peeked = value
                self.queue.append(it)                 # đưa về cuối hàng đợi để luân phiên

    def next(self):
        self._fill()
        value, self.peeked = self.peeked, _END
        return value

    def hasNext(self):
        self._fill()
        return self.peeked is not _END


z = ZigzagIterator([1, 2], [3, 4, 5, 6])
out = []
while z.hasNext():
    out.append(z.next())
print(out)                                            # [1, 3, 2, 4, 5, 6]

[1, 3, 2, 4, 5, 6]


## 16. So sánh các cách làm

| Cách | Ý tưởng | `__init__` | `next` / `hasNext` | Space phụ | Khi nào dùng |
|---|---|---|---|---|---|
| 1. Làm phẳng sẵn | copy mọi phần tử vào một danh sách | `O(N+R)` | `O(1)` | `O(N)` | đơn giản, dữ liệu nhỏ |
| 2. Hai con trỏ | `(i, j)` + `forward()` | `O(1)` | `O(1)` khấu hao | `O(1)` | **bản nộp chuẩn** |
| 3. Hai iterator | outer/inner + ô đệm | `O(1)` | `O(1)` khấu hao | `O(1)` | đúng tinh thần follow-up |
| 4. `chain` + lookahead | `chain.from_iterable` + ô đệm | `O(1)` | `O(1)` khấu hao | `O(1)` | ngắn nhất, Pythonic |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `list(chain.from_iterable([[1], [], [2, 3]]))`; (b) `next(iter([]), "x")`; (c) kết quả `run(Vector2D, ["Vector2D", "hasNext"], [[[[], []]], []])`; (d) sau `v = Vector2D([[1], [2]])` rồi `v.hasNext(); v.hasNext(); v.next()` trả bao nhiêu.

**Bài 2 (tính bằng tay).** Với `vec = [[], [4, 5], [], [6]]`, ghi `(i, j)` sau mỗi lời gọi của dãy `hasNext, next, next, hasNext, next, hasNext`.

**Bài 3 (điền TODO).** Hoàn thiện `Vector2DTodo` theo Cách 2.
#### Code demo

In [23]:
class Vector2DTodo:
    def __init__(self, vec):
        # TODO 1: lưu vec, và hai con trỏ i = 0, j = 0
        pass

    def _forward(self):
        # TODO 2: while i < len(vec) và j >= len(vec[i]): tăng i, đặt j = 0
        pass

    def next(self):
        # TODO 3: gọi _forward; lấy vec[i][j]; tăng j; trả giá trị
        pass

    def hasNext(self):
        # TODO 4: gọi _forward; trả i < len(vec)
        pass


# Tự làm xong rồi mới mở test:
# assert run(Vector2DTodo, examples_ops, examples_args) == examples_expected

**Bài 4 (tự viết lại).** Viết `Vector2DReverse` duyệt **ngược** (từ phần tử cuối của hàng cuối về phần tử đầu của hàng đầu) với `next`/`hasNext`, bằng con trỏ `(i, j)` đi lùi; kiểm tra với `model` đảo ngược.

**Bài 5 (biến thể / mini challenge).** Viết `Vector2DWithPeek` có thêm phương thức `peek()` (xem phần tử kế tiếp mà không tiến). Gợi ý: gọi `forward()` rồi đọc `vec[i][j]` mà **không** tăng `j`. Kiểm tra: `peek()` hai lần liên tiếp trả cùng một giá trị, và `next()` ngay sau đó trả giá trị đó.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `chain` bỏ qua hàng rỗng; `next(it, mặc_định)` không ném lỗi; `hasNext` không tiêu thụ. Bài 2: mỗi lời gọi trước tiên "nhảy qua hàng rỗng". Bài 4: đảo vai trò: bắt đầu `i = len - 1`, `j = len(vec[i]) - 1`.
- **Mức 2 (vừa):** Bài 1: `[1, 2, 3]`, `'x'`, `[None, False]`, `1`. Bài 2: bắt đầu `(0,0)`; `hasNext` → nhảy `(1,0)`; `next` → `4`, `(1,1)`; `next` → `5`, `(1,2)`; `hasNext` → nhảy `(3,0)`; `next` → `6`, `(3,1)`; `hasNext` → nhảy `(4,0)` rồi `False`. Bài 3: khung Cách 2.
- **Mức 3 (sâu):** Bài 4: `_backward`: `while i >= 0 and j < 0: i -= 1; j = len(vec[i]) - 1 nếu i >= 0`. Bài 5: `peek` giống `next` nhưng không `j += 1`; nhớ gọi `_forward` trước.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [24]:
# Bài 1
assert list(chain.from_iterable([[1], [], [2, 3]])) == [1, 2, 3]
assert next(iter([]), "x") == "x"
assert run(Vector2D, ["Vector2D", "hasNext"], [[[[], []]], []]) == [None, False]
v = Vector2D([[1], [2]])
v.hasNext()
v.hasNext()
assert v.next() == 1


# Bài 3
class Vector2DTodoAns:
    def __init__(self, vec):
        self.vec, self.i, self.j = vec, 0, 0

    def _forward(self):
        while self.i < len(self.vec) and self.j >= len(self.vec[self.i]):
            self.i += 1
            self.j = 0

    def next(self):
        self._forward()
        value = self.vec[self.i][self.j]
        self.j += 1
        return value

    def hasNext(self):
        self._forward()
        return self.i < len(self.vec)


assert run(Vector2DTodoAns, examples_ops, examples_args) == examples_expected

# Bài 2: ghi lại trạng thái bằng mô phỏng
w = Vector2DTodoAns([[], [4, 5], [], [6]])
states = []
for op in ["hasNext", "next", "next", "hasNext", "next", "hasNext"]:
    getattr(w, op)()
    states.append((w.i, w.j))
assert states == [(1, 0), (1, 1), (1, 2), (3, 0), (3, 1), (4, 0)]


# Bài 4
class Vector2DReverse:
    def __init__(self, vec):
        self.vec = vec
        self.i = len(vec) - 1
        self.j = len(vec[self.i]) - 1 if vec else -1

    def _backward(self):
        while self.i >= 0 and self.j < 0:
            self.i -= 1
            self.j = len(self.vec[self.i]) - 1 if self.i >= 0 else -1

    def next(self):
        self._backward()
        value = self.vec[self.i][self.j]
        self.j -= 1
        return value

    def hasNext(self):
        self._backward()
        return self.i >= 0


rng = random.Random(5)
for _ in range(200):
    vec = [[rng.randint(-5, 5) for _ in range(rng.randint(0, 3))] for _ in range(rng.randint(0, 5))]
    flat_rev = [x for row in vec for x in row][::-1]
    w = Vector2DReverse(vec)
    got = []
    while w.hasNext():
        got.append(w.next())
    assert got == flat_rev


# Bài 5
class Vector2DWithPeek(Vector2DTodoAns):
    def peek(self):
        self._forward()
        return self.vec[self.i][self.j]


w = Vector2DWithPeek([[], [7, 8]])
assert w.peek() == 7 and w.peek() == 7 and w.next() == 7 and w.next() == 8 and not w.hasNext()
print("PASS đáp án bài tập")

PASS đáp án bài tập


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (hai con trỏ + `forward`)

```python
class Vector2D:
    def __init__(self, vec):
        self.vec, self.i, self.j = vec, 0, 0

    def next(self) -> int:
        self.forward()
        ans = self.vec[self.i][self.j]
        self.j += 1
        return ans

    def hasNext(self) -> bool:
        self.forward()
        return self.i < len(self.vec)

    def forward(self):
        while self.i < len(self.vec) and self.j >= len(self.vec[self.i]):
            self.i += 1
            self.j = 0
```

**Checklist**

- [ ] Có hàm "tiến tới ô thật" (`forward`) dùng `while` (bỏ qua **nhiều** hàng rỗng) và kiểm tra `i < len(vec)` **trước** khi đọc `vec[i]`?
- [ ] Cả `next` **và** `hasNext` đều gọi `forward`?
- [ ] `hasNext` không tiêu thụ/không đổi phần tử (gọi nhiều lần vẫn như một lần)?
- [ ] Không sửa dữ liệu đầu vào (`pop`, `del`)?
- [ ] Đã thử `[]`, `[[]]`, hàng rỗng ở đầu/giữa/cuối, một phần tử?
- [ ] Đã dựng `run(cls, ops, args)` và so với mô hình brute force trên thao tác ngẫu nhiên?
- [ ] Skeleton đúng `class Vector2D` / `__init__(self, vec)` / `next(self) -> int` / `hasNext(self) -> bool`?